In [4]:
import pandas as pd

df = pd.read_csv(r'C:\WC26\data\results.csv')
df['date'] = pd.to_datetime(df['date'])

# keep post-2010 only — modern football is more relevant
df = df[df['date'] >= '2010-01-01'].reset_index(drop=True)

print(df.shape)
print(df.head())

(15889, 9)
        date home_team    away_team  home_score  away_score tournament  \
0 2010-01-02      Iran  North Korea         1.0         0.0   Friendly   
1 2010-01-02     Qatar         Mali         0.0         0.0   Friendly   
2 2010-01-02     Syria     Zimbabwe         6.0         0.0   Friendly   
3 2010-01-02     Yemen   Tajikistan         0.0         1.0   Friendly   
4 2010-01-03    Angola       Gambia         1.0         1.0   Friendly   

                         city   country  neutral  
0                        Doha     Qatar     True  
1                        Doha     Qatar    False  
2                Kuala Lumpur  Malaysia     True  
3                      Sana'a     Yemen    False  
4  Vila Real de Santo António  Portugal     True  


In [5]:
def get_outcome(row):
    if row['home_score'] > row['away_score']:
        return 2   # home win
    elif row['home_score'] < row['away_score']:
        return 0   # away win
    else:
        return 1   # draw

df['outcome'] = df.apply(get_outcome, axis=1)

In [6]:
from collections import defaultdict

elo_ratings = defaultdict(lambda: 1500)  # all teams start at 1500
K = 32  # update speed

def expected_score(ra, rb):
    return 1 / (1 + 10 ** ((rb - ra) / 400))

def update_elo(winner_elo, loser_elo, result, k=K):
    # result: 1 = win, 0.5 = draw, 0 = loss
    exp = expected_score(winner_elo, loser_elo)
    new = winner_elo + k * (result - exp)
    return round(new, 2)

elo_before_home = []
elo_before_away = []

df = df.sort_values('date').reset_index(drop=True)

for _, row in df.iterrows():
    h, a = row['home_team'], row['away_team']
    eh, ea = elo_ratings[h], elo_ratings[a]

    elo_before_home.append(eh)
    elo_before_away.append(ea)

    if row['outcome'] == 2:    # home win
        elo_ratings[h] = update_elo(eh, ea, 1)
        elo_ratings[a] = update_elo(ea, eh, 0)
    elif row['outcome'] == 0:  # away win
        elo_ratings[h] = update_elo(eh, ea, 0)
        elo_ratings[a] = update_elo(ea, eh, 1)
    else:                      # draw
        elo_ratings[h] = update_elo(eh, ea, 0.5)
        elo_ratings[a] = update_elo(ea, eh, 0.5)

df['elo_home'] = elo_before_home
df['elo_away'] = elo_before_away
df['elo_diff'] = df['elo_home'] - df['elo_away']

In [7]:
def recent_form(team, date, df, n=5):
    past = df[(df['date'] < date) & 
              ((df['home_team'] == team) | (df['away_team'] == team))].tail(n)
    points = 0
    for _, r in past.iterrows():
        if r['home_team'] == team:
            if r['outcome'] == 2: points += 3
            elif r['outcome'] == 1: points += 1
        else:
            if r['outcome'] == 0: points += 3
            elif r['outcome'] == 1: points += 1
    return points  # max 15

df['form_home'] = df.apply(lambda r: recent_form(r['home_team'], r['date'], df), axis=1)
df['form_away'] = df.apply(lambda r: recent_form(r['away_team'], r['date'], df), axis=1)
df['form_diff'] = df['form_home'] - df['form_away']

In [11]:
df['is_neutral'] = df['neutral'].astype(int)

tournament_weight = {
    'FIFA World Cup': 3,
    'UEFA Euro': 2,
    'Copa América': 2,
    'AFC Asian Cup': 2,
    'Friendly': 0.5,
}
df['tournament_weight'] = df['tournament'].map(tournament_weight).fillna(1)

In [12]:
features = ['elo_diff', 'form_diff', 'is_neutral', 'tournament_weight']
target = 'outcome'

df[features + [target, 'date', 'home_team', 'away_team']].to_csv(r'C:\WC26\data\wc_features.csv', index=False)
print("Done. Shape:", df.shape)
print(df[features].describe())

Done. Shape: (15889, 18)
           elo_diff     form_diff    is_neutral  tournament_weight
count  15889.000000  15889.000000  15889.000000       15889.000000
mean      14.214421      0.214740      0.307131           0.914375
std      151.155995      4.732239      0.461319           0.433358
min     -837.680000    -15.000000      0.000000           0.500000
25%      -69.810000     -3.000000      0.000000           0.500000
50%       13.200000      0.000000      0.000000           1.000000
75%       97.940000      3.000000      1.000000           1.000000
max      775.790000     15.000000      1.000000           3.000000
